# 🌾 Kaggriculture: 6 Engine Traps, Each One Measured

I lost three agents to this engine before I stopped guessing and started reading `kaggriculture.py`. Every trap below is one I actually hit, and **each one is measured here by running the real engine** — no claims from memory, no hand-waving.

The headline: **the Wheat Loop example in the official getting-started guide harvests too early and leaves about a third of the yield in the field.** That is not a criticism of the docs — it is a minimal example doing its job. But if you built your first agent from it (I did), you inherited the bug.

**What's in here**
1. Reading the engine's own numbers instead of trusting a table
2. Trap 1 — the planting day already counts as an unwatered day
3. Trap 2 — the watering window: *when* you water decides your yield
4. Trap 3 — harvest timing, and the 33% the starter leaves behind
5. Trap 4 — the expiry cliff: one day late and you get **zero**
6. Trap 5 — fertilizer only pays on watered days (+67%)
7. Trap 6 — the shed is not a tile, and other geography traps
8. Putting honest numbers on the crops
9. A pre-flight checklist

Everything is CPU-only and runs in about a minute. **No strategy here — this is all engine mechanics**, the stuff that is true no matter what your agent does with it.

# 1. Setup — read the engine, don't trust a table

The single most useful habit in this competition: the environment ships its own source. Every constant below is imported live from the installed package, so this notebook can never drift out of date relative to the engine it is describing.

In [ ]:
import sys, io, contextlib, subprocess

def _quiet_import():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        import kaggle_environments
        from kaggle_environments import make
        from kaggle_environments.envs.kaggriculture import kaggriculture as K
    return kaggle_environments, make, K

try:
    kaggle_environments, make, K = _quiet_import()
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle-environments'])
    kaggle_environments, make, K = _quiet_import()

print('kaggle-environments', getattr(kaggle_environments, '__version__', 'unknown'))
print('engine file:', K.__file__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110
BLUE = '#2a78d6'
GREY = '#b8c4d0'

crops = pd.DataFrame(K.CROPS).T
crops.index.name = 'crop'
crops

In [ ]:
animals = pd.DataFrame(K.ANIMALS).T
animals.index.name = 'animal'
print(animals.to_string())
print()
print('starting money      :', 3000)
print('season              :', '24 turns/day x 30 days = 720 turns')
print('shed access tiles   :', K._shed_access_tiles(10))
print('land prices (NE/SW/SE):', K.LAND_PRICES)
print('hire cost sequence  :', [K._fib(n) for n in range(9)])

#### Observation
Two things already stand out and both cost me an agent later:

* `LAND_PRICES` is `[1000, 2000, 4000]` against a `$3000` starting bank. You cannot buy your way to a full board early — the board *has* to be earned.
* The hire cost is a Fibonacci sequence **that resets every day**. Eight hands in one day cost `1+1+2+3+5+8+13+21 = $54`, which is cheap. The ninth alone costs `$34`. Labour is nearly free up to a point and then bites hard.

# 2. Trap 1 — the planting day is already an unwatered day

Here is the constructor for a new plant, straight from the engine:

In [ ]:
import inspect
src = inspect.getsource(K._new_plant)
print(src)

Note `"consecutive_unwatered": 1` with the comment *"planting day counts as unwatered"*. Now the end-of-day refresh:

In [ ]:
ref = inspect.getsource(K._daily_refresh_plants)
print('\n'.join(ref.splitlines()[:22]))

Put those together. A plant is born at `consecutive_unwatered = 1`. At the end of that same day, if it was not watered, the counter goes to `2`, and `>= 2` turns the tile into a weed.

**So a seed planted and not watered on its planting day is dead by midnight — you do not get a day of grace.** Let's prove it on the real engine rather than take my word for it.

In [ ]:
def observe_tile(obs):
    me = obs['farms'][obs['player']]
    fx, fy = me['farmer']
    return me['tiles'][fy][fx]

def play(policy, days=20, seed=7, log_fn=None):
    """Run one episode; player 1 passes. Returns the log list."""
    log = []
    buf = io.StringIO()
    def p0(obs, cfg):
        if log_fn: log_fn(obs, log)
        return policy(obs)
    def p1(obs, cfg):
        return {'farmer': ['PASS'], 'hands': [], 'market': []}
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        env = make('kaggriculture',
                   configuration={'episodeSteps': days * 24, 'seed': seed}, debug=True)
        env.run([p0, p1])
    return log

def log_tile(obs, log):
    t = observe_tile(obs)
    rec = {'step': obs['step'], 'day': obs['day']}
    if isinstance(t, dict):
        rec.update({k: t.get(k) for k in
                    ('kind', 'crop', 'yield_units', 'consecutive_unwatered', 'watered_today')})
    else:
        rec['kind'] = t
    log.append(rec)

def pol_never_water(obs):
    me, priv = obs['farms'][obs['player']], obs['private']
    tile = observe_tile(obs)
    mk = []
    if priv['seeds'].get('WHEAT', 0) == 0 and me['money'] >= 10:
        mk.append(['BUY_SEED', 'WHEAT', 1])
    act = ['PASS']
    if tile is None and priv['seeds'].get('WHEAT', 0) > 0 and obs['day'] == 0:
        act = ['PLANT', 'WHEAT']
    return {'farmer': act, 'hands': [], 'market': mk}

log = play(pol_never_water, days=4, log_fn=log_tile)
df = pd.DataFrame(log)
planted = df[df.kind == 'PLANT']
weeded  = df[df.kind == 'WEED']
print(f"planted at step {planted.step.min()} (day {planted.day.min()})")
print(f"survived {len(planted)} turns")
print(f"became a WEED at step {weeded.step.min()} (day {weeded.day.min()})")
df.head(3)

#### Observation — confirmed, and it is brutal

The seed goes in on **step 2 of day 0** and the tile is a **weed by step 24**, the very first end-of-day refresh. It never saw day 1.

This is the single most common reason a beginner agent looks like it is doing everything right and still earns nothing: it plants a whole field in one sweep, then starts watering tomorrow. By tomorrow the field is weeds. **Water on the planting day, always** — and size your planting to the number of units you can actually water that same day. Your labour supply, not your seed budget, is what caps how fast you can plant.

# 3. Trap 2 — the watering window decides your yield

Most people read "water your plants daily or they die" and stop there. But for one-time crops (wheat, carrot, melon) watering does something else entirely — it is *the* mechanism that accumulates harvestable yield, and only inside a specific window. Here is the `WATER` branch of the engine:

In [ ]:
step_src = inspect.getsource(K)
i = step_src.index('if op == "WATER"')
print(step_src[i:i + 640])

Read the window carefully:

```
window_start = (max_yield_day + 1) // 2      # i.e. ceil(max_yield_day / 2)
if window_start <= age_days <= max_yield_day:
    yield_units += 1        (+2 if fertilized)
```

So watering **outside** that window keeps the plant alive but adds **nothing** to the harvest. And the number of units you can ever get is therefore set by how many days of the window you actually water through — *not* by the crop's `max_yield` field.

In [ ]:
rows = []
for c, cd in K.CROPS.items():
    if cd['ongoing']:
        continue
    ws = (cd['max_yield_day'] + 1) // 2
    we = cd['max_yield_day']
    n_water = we - ws + 1
    rows.append({
        'crop': c,
        'first_yield_day': cd['first_yield_day'],
        'max_yield_day': we,
        'window': f'days {ws}-{we}',
        'waterings_in_window': n_water,
        'engine max_yield': cd['max_yield'],
        'reachable by watering': min(cd['max_yield'], 1 + n_water),
        'reachable +fertilizer': min(cd['max_yield'], 1 + 2 * n_water),
    })
win = pd.DataFrame(rows).set_index('crop')
win

#### Observation — `max_yield` is a cap, not a target

Look at **wheat**: the engine advertises `max_yield = 6`, but the watering window is only 3 days long, and a plant starts at 1 unit. So `1 + 3 = 4` is the most you can ever harvest **without fertilizer**. The `6` is unreachable by watering alone.

Carrot is the same story (`4` advertised, `3` reachable). Melon is the exception — its window is 7 days long, so watering alone saturates its cap of 6.

If you sized your economics off the `max_yield` column, every wheat and carrot number you computed was **50% too optimistic**. I did exactly this.

# 4. Trap 3 — harvest timing, and the 33% the starter leaves behind

Now the practical consequence. The official Wheat Loop example harvests as soon as it legally can:

```python
crop_age = obs["day"] - tile["planted_day"]
if crop_age >= 2:      # WHEAT first_yield_day = 2
    return {"farmer": ["HARVEST"], ...}
```

But wheat's bonus window is days **2 to 4**. Harvesting on day 2 collects the plant at 1 unit and throws away the two bonus waterings still available. Waiting trades cycle *speed* for units *per cycle* — so which wins?

I ran the controlled experiment below. The farmer spawns on `(4,4)`, which is conveniently both plantable and shed-adjacent, so one tile can be planted, watered, harvested and emptied **without the farmer ever moving.** That removes walking time from the comparison entirely.

In [ ]:
def single_tile(crop, harvest_age, water=True, fertilize=False, days=20, seed=7):
    """Farm one tile for `days` days; return units harvested and per-cycle yields."""
    st = {'units': 0, 'prev': 0, 'cycles': 0, 'yields': []}
    def pol(obs):
        me, priv = obs['farms'][obs['player']], obs['private']
        tile = observe_tile(obs)
        inv = (priv['inventories'][0] or {}).get(crop, 0)
        if inv > st['prev']:
            st['units'] += inv - st['prev']; st['yields'].append(inv - st['prev']); st['cycles'] += 1
        st['prev'] = inv
        mk = []
        if priv['seeds'].get(crop, 0) == 0 and me['money'] >= 200:
            mk.append(['BUY_SEED', crop, 1])
        if fertilize and priv['shed'].get('FERTILIZER', 0) == 0 and me['money'] >= 200:
            mk.append(['BUY_PRODUCT', 'FERTILIZER', 1])
        act = ['PASS']
        fert_inv = (priv['inventories'][0] or {}).get('FERTILIZER', 0)
        if inv >= 12:
            act = ['DROP']; st['prev'] = 0
        elif tile is None and priv['seeds'].get(crop, 0) > 0:
            act = ['PLANT', crop]
        elif isinstance(tile, dict) and tile.get('kind') == 'WEED':
            act = ['DIG']
        elif isinstance(tile, dict) and tile.get('kind') == 'PLANT':
            age = obs['day'] - tile['planted_day']
            if age >= harvest_age and tile.get('yield_units', 0) > 0:
                act = ['HARVEST']
            elif fertilize and fert_inv == 0 and priv['shed'].get('FERTILIZER', 0) > 0:
                act = ['PICKUP', 'FERTILIZER', 1]
            elif fertilize and fert_inv > 0 and tile.get('fertilized_until_day', -1) < obs['day']:
                act = ['FERTILIZE']
            elif water and not tile['watered_today']:
                act = ['WATER']
        return {'farmer': act, 'hands': [], 'market': mk}
    play(pol, days=days, seed=seed)
    return st

res = {}
for age in (2, 3, 4, 5, 6):
    s = single_tile('WHEAT', age, days=20)
    per = s['yields'][1:] or s['yields']
    res[age] = {'cycles': s['cycles'],
                'units_per_cycle': round(np.mean(per), 2) if per else 0.0,
                'total_units_20d': s['units']}
harvest_df = pd.DataFrame(res).T
harvest_df.index.name = 'harvest at age >='
harvest_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ages = list(harvest_df.index)
cols = [BLUE if a != 2 else '#d9534f' for a in ages]
axes[0].bar([str(a) for a in ages], harvest_df.units_per_cycle, color=cols)
axes[0].set_title('Units per harvest'); axes[0].set_xlabel('harvest at crop age >=')
axes[1].bar([str(a) for a in ages], harvest_df.total_units_20d, color=cols)
axes[1].set_title('Total units over 20 days'); axes[1].set_xlabel('harvest at crop age >=')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Wheat, one tile, no movement   (red = what the docs example does)', y=1.04)
plt.tight_layout(); plt.show()

a2, a3 = harvest_df.total_units_20d[2], harvest_df.total_units_20d[3]
print(f'harvest at age 2 (docs): {a2} units / 20 days')
print(f'harvest at age 3       : {a3} units / 20 days   ->  {(a3/a2-1)*100:+.0f}%')

#### Observation — wait one more day, get a third more wheat

Harvesting at age 2 gives exactly **1 unit per cycle**, nine times over twenty days: **9 units**. Waiting to age 3 gives **2 units** per cycle, six times: **12 units**. Same tile, same labour, **+33%** — from deleting one character in a comparison.

Notice the shape of the rest of the curve. Units per cycle rise exactly `+1` per extra day held (1, 2, 3, 4), which is the watering window doing its work, one bonus per day. But total units **plateau at 12** from age 3 onward, because longer holds mean fewer cycles. Age 3 reaches that plateau with the fastest turnover, which also leaves the most slack for everything else your units need to do.

And then age 6 collapses to **zero**. That is the next trap.

# 5. Trap 4 — the expiry cliff: one day late and you get nothing

Look again at `_new_plant`: a one-time crop is stamped with

```python
max_lifespan_step = (day + max_yield_day + 1) * turns_per_day
```

For wheat planted on day 0 that is step `(0 + 4 + 1) * 24 = 120`, the start of day 5. After that, `_decay_plants` runs:

In [ ]:
print(inspect.getsource(K._decay_plants))

Past its lifespan the plant sheds **one yield unit every other turn**, and the moment it hits zero the tile becomes a weed. A wheat plant carrying 4 units is stripped bare in 8 turns — a third of a day.

That is why the age-6 row above harvested **0 units in 20 days**: the agent was always waiting for a day that arrived after the plant had already rotted. It looks like a perfectly reasonable "let it grow a bit longer" policy, and it earns exactly nothing.

In [ ]:
decay_log = []
def pol_watch_decay(obs):
    me, priv = obs['farms'][obs['player']], obs['private']
    tile = observe_tile(obs)
    mk = []
    if priv['seeds'].get('WHEAT', 0) == 0 and me['money'] >= 10:
        mk.append(['BUY_SEED', 'WHEAT', 1])
    act = ['PASS']
    if tile is None and priv['seeds'].get('WHEAT', 0) > 0 and obs['day'] == 0:
        act = ['PLANT', 'WHEAT']
    elif isinstance(tile, dict) and tile.get('kind') == 'PLANT':
        if not tile['watered_today']:
            act = ['WATER']          # water forever, never harvest
    if isinstance(tile, dict):
        decay_log.append({'step': obs['step'], 'day': obs['day'],
                          'kind': tile.get('kind'), 'units': tile.get('yield_units')})
    return {'farmer': act, 'hands': [], 'market': mk}

play(pol_watch_decay, days=7)
dd = pd.DataFrame(decay_log)
peak = dd.units.max()
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(dd.step, dd.units.fillna(0), color=BLUE, lw=2)
ax.axvline(120, color='#d9534f', ls='--', lw=1.2)
ax.text(122, peak * 0.6, 'max_lifespan_step = 120\n(start of day 5)', color='#d9534f', fontsize=9)
ax.set_title('A wheat plant that is watered forever and never harvested')
ax.set_xlabel('step'); ax.set_ylabel('yield_units')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
rot = dd[dd.kind == 'WEED']
print(f'peak units held        : {peak:.0f}')
print(f'tile became a WEED at  : step {rot.step.min()} (day {rot.day.min()})')
print(f'units salvaged         : 0  -- the whole harvest rotted on the tile')

#### Observation
The plant climbs to its 4-unit ceiling through the watering window, sits there, and then falls off a cliff at exactly step 120 — the value the engine wrote into the tile the moment it was planted. **Your harvest deadline is fixed at planting time**, and it is one day after `max_yield_day`, not some vague "when it looks ripe".

Practical rule: for a one-time crop, the safe harvest day is anywhere in `[first_yield_day, max_yield_day]`, and you want to be at the top of that range but never past it.

# 6. Trap 5 — fertilizer only pays on watered days

`FERTILIZE` is active for the current day plus two more, and inside the watering window it doubles the bonus (`+2` instead of `+1`). But re-read the `WATER` branch: the doubling is applied **inside the watering code**. Fertilizer on an unwatered plant does nothing at all — the engine comments this explicitly as "basic needs first".

For wheat this is the only way to reach the advertised `max_yield` of 6.

In [ ]:
plain = single_tile('WHEAT', 4, water=True, fertilize=False, days=20)
fert  = single_tile('WHEAT', 4, water=True, fertilize=True,  days=20)
dry   = single_tile('WHEAT', 4, water=False, days=20)

comp = pd.DataFrame({
    'watered only':        [plain['units'], plain['cycles']],
    'watered + fertilized':[fert['units'],  fert['cycles']],
    'never watered':       [dry['units'],   dry['cycles']],
}, index=['units in 20 days', 'harvests']).T
print(comp.to_string())
print(f"\nfertilizer effect: {plain['units']} -> {fert['units']} units "
      f"({(fert['units']/plain['units']-1)*100:+.0f}%)")

#### Observation
Fertilizer lifts the same tile from **12 to 20 units** over twenty days — **+67%**, comfortably more than the +33% from fixing harvest timing. And the "never watered" column is the control: **0 units**, because as we proved in Trap 1 those plants die on their planting day.

Worth knowing where fertilizer comes from: animals produce it via `COLLECT_FERTILIZER` (1 per animal per day), and it is one of only two products you can buy back from the market with `BUY_PRODUCT` — wheat being the other. That connection between the animal side and the crop side is easy to miss if you treat them as separate subsystems.

# 7. Trap 6 — the shed is not a tile, and other geography traps

Three map facts that each cost me debugging time:

**The shed does not exist in `tiles`.** You will search the grid for it and find nothing. It is a virtual store you can reach only by standing on one of four specific tiles:

In [ ]:
print('shed access tiles :', K._shed_access_tiles(10))
print('farmer spawn      :', K._default_spawn(10))
print()
print(inspect.getsource(K._initial_tile))
print(inspect.getsource(K._quadrant_of))

**Only the NW quadrant starts unlocked.** `_initial_tile` marks every other tile `"LOCKED"`. Units may *walk across* locked ground but cannot act on it, so a policy that scans the whole 10×10 grid for empty tiles will happily send a hand to a tile it can never plant, every single turn, forever. That was my second dead agent.

**Hiring resets daily and is convex.** `hires_today` drives the cost through `fib`, so the price you pay depends only on how many you already hired *today*: 

In [ ]:
costs = [K._fib(n) for n in range(10)]
cum = np.cumsum(costs)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(1, 11), costs, color=BLUE, label='cost of the n-th hire')
ax.plot(range(1, 11), cum, color='#d9534f', marker='o', ms=4, label='cumulative cost that day')
ax.set_xlabel('n-th hire of the day'); ax.set_ylabel('$')
ax.set_title('Hiring is nearly free, then suddenly is not')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
for n in (4, 6, 8, 10):
    print(f'{n:2d} hands in one day costs ${cum[n-1]:,d}')

#### Observation
Six hands cost **$20** for the whole day, eight cost **$54**, ten cost **$143**. Cheap enough that under-hiring is almost always the more expensive mistake — remember Trap 1: an unwatered plant is a *dead* plant, so labour shortage does not slow you down, it destroys inventory you already paid for.

A related gotcha: `FEED` consumes one `WHEAT` **from the inventory of the unit performing the action**, not from the shed. A hand standing next to a hungry cow with an empty satchel silently does nothing.

# 8. Putting honest numbers on the crops

Now we can build a crop table that uses **reachable** yield instead of the advertised cap. I will price everything at `base`, which is the market price when inventory sits at the shared starting level `I0`.

In [ ]:
rows = []
for c, cd in K.CROPS.items():
    base = K.MARKET_PARAMS[c]['base']
    if cd['ongoing']:
        units = cd['max_yield']
        span  = cd['first_yield_day'] + cd['interval'] * (cd['max_yield'] - 1) + 1
        note  = 'ongoing'
    else:
        ws    = (cd['max_yield_day'] + 1) // 2
        units = min(cd['max_yield'], 1 + (cd['max_yield_day'] - ws + 1))
        span  = cd['max_yield_day'] + 1
        note  = 'one-time'
    rev  = units * base
    prof = rev - cd['seed']
    rows.append({'crop': c, 'type': note, 'seed $': cd['seed'], 'tile-days': span,
                 'units (reachable)': units, 'base $/unit': base,
                 'gross $': rev, 'profit $': prof,
                 '$ per tile-day': round(prof / span, 1)})
econ = pd.DataFrame(rows).sort_values('$ per tile-day', ascending=False).set_index('crop')
econ

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
e = econ.sort_values('$ per tile-day')
ax.barh(e.index, e['$ per tile-day'], color=BLUE)
ax.set_title('Profit per tile-day at base prices (reachable yield, seed cost deducted)')
ax.set_xlabel('$ per tile-day')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

#### Observation — and an important caveat

At base prices melon dominates on profit per tile-day by a wide margin. But **do not read this table as a strategy**, for three reasons the table cannot show:

1. **Capital lockup.** Melon costs `$80` a tile and returns nothing for ten days. Against a `$3000` bank, filling tiles with melon on day one spends the purse before a single coin comes back — and a farm that cannot pay for hands cannot water, and unwatered plants die (Trap 1). The naive ranking quietly assumes money is free.
2. **The market fights back.** Prices are a function of market inventory, and the response is asymmetric per product. Look at `above_target` in `MARKET_PARAMS`: melon is `3.60` with a `sq` shape and wool is `3.20`, while wheat is `0.20` with a `log` shape. Premium goods collapse toward the `$1` floor when you dump them; staples barely move. A per-unit base price is the price of the *first* unit you sell, not the hundredth.
3. **Labour.** Every one of those units needs a unit standing on the tile to water and harvest it, and both players draw from the same 24-turn day.

So this table tells you what a tile *can* yield, not what your farm *should* plant. Where those three constraints bind against each other is exactly where agent design actually happens — and that part everyone has to work out for themselves.

In [ ]:
mp = pd.DataFrame(K.MARKET_PARAMS).T[['base', 'T', 'below_func', 'below_target',
                                       'above_func', 'above_target']]
mp.index.name = 'product'
print('How hard the price falls when you oversupply (above_* columns):')
mp.sort_values('above_target', ascending=False)

# 9. Pre-flight checklist

Everything above, condensed into the checks I now run against any agent before submitting it:

| # | Check | Why |
|---|---|---|
| 1 | Does every seed get watered **on its planting day**? | Otherwise it is a weed by midnight — proven in Trap 1 |
| 2 | Do you plant only as many tiles as you can water that same day? | Planting capacity is bounded by labour, not by seed money |
| 3 | Is your harvest age at the **top** of `[first_yield_day, max_yield_day]`? | Worth +33% on wheat vs. harvesting immediately |
| 4 | Do you ever harvest **later** than `max_yield_day`? | Past `max_lifespan_step` the plant rots to nothing in under a day |
| 5 | Are your yield estimates using reachable units, not `max_yield`? | Wheat reaches 4 of its advertised 6 without fertilizer |
| 6 | Does fertilizer only go on plants you are also watering? | Unfertilized-but-watered beats fertilized-but-dry every time |
| 7 | Do you skip `"LOCKED"` tiles when scanning for work? | Units can walk there but never act there |
| 8 | Do units carry `WHEAT` before you ask them to `FEED`? | `FEED` draws from the acting unit's own inventory |
| 9 | Are you shed-adjacent before `DROP` / `PICKUP`? | The shed is not in `tiles`; only 4 tiles reach it |
| 10 | Are you selling everything before the final step? | Unsold stock scores zero |

The one that surprises people most is #3, because the official example does the opposite and nothing about it looks wrong. Fixing it was the single cheapest improvement I made — one comparison operator, a third more wheat.

## Try it yourself

Everything here is reproducible in a minute: change `harvest_age` in the `single_tile` function above, or point it at `CARROT` or `MELON` and watch the window shift. The engine source is right there in your environment at the path printed in section 1 — reading it is genuinely the highest-return hour you can spend on this competition.

Good luck out there. 🌾

**If this saved you a debugging session, an upvote is appreciated 🙌 — and if you hit a trap I missed, please leave it in the comments so the list grows.**